In [2]:
# Install dependencies if necessary
# %pip install opencv-python numpy

import os
import cv2
import numpy as np
from pathlib import Path

# ==========================================
# CHANGE THIS PATH TO YOUR .npy FILES FOLDER
# ==========================================
# This can be a direct path to a word folder (e.g., 'processed_webcam_data/landmarks/HELLO')
# or a main folder containing subfolders of words (e.g., 'processed_webcam_data/landmarks')
INPUT_NPY_PATH = "processed_webcam_data/landmarks"

OUTPUT_RECONSTRUCT_PATH = "reconstruction"

# Constants
FPS = 30
WIDTH = 640
HEIGHT = 480

def reconstruct(data, outfile):
    writer = cv2.VideoWriter(str(outfile), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
    for i, frame in enumerate(data):
        canvas = np.full((HEIGHT, WIDTH, 3), 255, dtype=np.uint8)
        if np.all(frame == 0):
            cv2.putText(canvas, "PAD", (280, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
            writer.write(canvas)
            continue

        # Extract landmarks from the flat array
        pose = frame[:15].reshape(5, 3)
        lh = frame[15:78].reshape(21, 3)
        rh = frame[78:].reshape(21, 3)

        def draw(points, color):
            for p in points:
                if p[0] == 0 and p[1] == 0 and p[2] == 0:
                    continue
                x = int((p[0] * 0.25 + 0.5) * WIDTH)
                y = int((p[1] * 0.25 + 0.5) * HEIGHT)
                cv2.circle(canvas, (x, y), 3, color, -1)

        draw(pose, (0, 0, 255))
        draw(lh, (0, 255, 0))
        draw(rh, (255, 255, 0))

        cv2.putText(canvas, f"Frame {i}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 0, 0), 2)
        writer.write(canvas)
    writer.release()

def main():
    input_path = Path(INPUT_NPY_PATH)
    output_path = Path(OUTPUT_RECONSTRUCT_PATH)
    
    if not input_path.exists():
        print(f"Error: Input path '{input_path}' does not exist.")
        return
        
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"Saving reconstructions to: {output_path.absolute()}")
    
    # Recursively find all .npy files
    npy_files = list(input_path.rglob("*.npy"))
    
    if not npy_files:
        print(f"No .npy files found in '{input_path}'.")
        return
        
    print(f"Found {len(npy_files)} .npy files to reconstruct.")
    
    for npy_file in npy_files:
        # Load the data
        data = np.load(npy_file)
        
        # Keep folder structure relative to input path if possible
        try:
            rel_path = npy_file.relative_to(input_path)
            if rel_path.parent == Path('.'):
                out_dir = output_path
            else:
                out_dir = output_path / rel_path.parent
        except ValueError:
            out_dir = output_path
            
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{npy_file.stem}.mp4"
        
        print(f"Reconstructing {npy_file.name} -> {out_file}")
        reconstruct(data, out_file)
        
    print("\nAll reconstructions complete!")

if __name__ == "__main__":
    main()


Saving reconstructions to: d:\BRIDGING SILENCE DATA COLLECTION PIPELINE\reconstruction
Found 7 .npy files to reconstruct.
Reconstructing sample_1782131548.npy -> reconstruction\HELLO\sample_1782131548.mp4
Reconstructing sample_1782131561.npy -> reconstruction\HELLO\sample_1782131561.mp4
Reconstructing sample_1782131574.npy -> reconstruction\HELLO\sample_1782131574.mp4
Reconstructing sample_1782132470.npy -> reconstruction\HELLO\sample_1782132470.mp4
Reconstructing sample_1782133037.npy -> reconstruction\HELLO\sample_1782133037.mp4
Reconstructing sample_1782133377.npy -> reconstruction\HELLO\sample_1782133377.mp4
Reconstructing sample_1782133409.npy -> reconstruction\yikes\sample_1782133409.mp4

All reconstructions complete!
